# Molecular Biology and the Flow of Genetic Information Workflow

This notebook scaffold supports the article **Molecular Biology and the Flow of Genetic Information**. It can be expanded with sequence comparison, transcript kinetics, expression matrices, codon usage, translation, mutation-rate estimation, molecular flow scoring, and provenance notes.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

article_dir = Path.cwd().parent
decay = pd.read_csv(article_dir / 'data' / 'transcript_decay.csv')
slope, intercept = np.polyfit(decay['time_h'], np.log(decay['expression']), 1)
k_est = -slope
half_life_h = np.log(2) / k_est
auc = np.trapz(decay['expression'], decay['time_h'])
pd.DataFrame({'k_est':[k_est], 'half_life_h':[half_life_h], 'AUC':[auc]}).round(4)

In [ ]:
expr = pd.read_csv(article_dir / 'data' / 'expression_matrix.csv').set_index('gene')
metadata = pd.read_csv(article_dir / 'data' / 'sample_metadata.csv')
control = metadata.loc[metadata['group'] == 'control', 'sample'].tolist()
treated = metadata.loc[metadata['group'] == 'treated', 'sample'].tolist()
summary = pd.DataFrame(index=expr.index)
summary['control_mean'] = expr[control].mean(axis=1)
summary['treated_mean'] = expr[treated].mean(axis=1)
summary['log2_fc'] = np.log2((summary['treated_mean'] + 1) / (summary['control_mean'] + 1))
summary.sort_values('log2_fc', ascending=False).round(4)

In [ ]:
coding_seq = (article_dir / 'data' / 'coding_sequence.txt').read_text().strip().upper()
codons = [coding_seq[i:i+3] for i in range(0, len(coding_seq)-2, 3)]
gc_fraction = sum(base in {'G', 'C'} for base in coding_seq) / len(coding_seq)
pd.Series(codons).value_counts().rename_axis('codon').reset_index(name='count').assign(gc_fraction=gc_fraction).head(20)

In [ ]:
mut = pd.read_csv(article_dir / 'data' / 'mutation_observations.csv')
mut['mutation_rate'] = mut['observed_mutations'] / (mut['genomes_surveyed'] * mut['sites_surveyed'] * mut['generations'])
mut.sort_values('mutation_rate', ascending=False)

In [ ]:
condition = pd.read_csv(article_dir / 'data' / 'molecular_flow_condition_sites.csv')
condition['molecular_flow_score'] = (
    0.16 * condition['replication_fidelity'] +
    0.15 * condition['transcription_signal'] +
    0.14 * condition['rna_processing'] +
    0.14 * condition['translation_support'] +
    0.16 * condition['repair_capacity'] +
    0.15 * condition['regulatory_context'] +
    0.10 * (1 - condition['expression_noise_risk'])
)
condition.sort_values('molecular_flow_score', ascending=False).round(3)